# CS769 - Experiment 5 (Kaggle): Post-Merge Repair Budget Sweep

**Settings:** Accelerator = *GPU T4 x2*, Internet = *On*.

| Budget ($k$) | Pipeline | DPO Data | Optimizer Steps (batch 1 $\times$ accum 8) |
|---|---|---|---|
| $0$ | `W1` (unaligned merge baseline) | none | 0 |
| $125$ | `W1` $\rightarrow$ DPO | 125 nested pairs | ~16 (~2 min) |
| $500$ | `W1` $\rightarrow$ DPO | 500 nested pairs | ~63 (~8 min) |
| $2000$ | `W3` (full post-merge DPO) | 2,000 full pool | 250 (reuses E4) |

**Merges tested:** Linear, TIES (density 0.2), DARE-TIES (drop 0.5, density 0.2, seed 10), equal weights $1/3$.

In [ ]:
!git clone https://github.com/abhinavpareek655/LLM_Merging.git
!nvidia-smi -L
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -U peft datasets accelerate matplotlib
!pip uninstall -y torchao
import transformers, peft; print(transformers.__version__, peft.__version__)

In [ ]:
import os
REPO = "/kaggle/working/LLM_Merging"
os.chdir(REPO)

need = [
    "src/eval/eval_math.py", "src/eval/eval_code.py", "src/eval/eval_ifeval.py",
    "src/train/train_dpo_stable.py", "src/data/precompute_ref_logps.py", "configs/base_config.py",
    "outputs/datasets/e_pref_train.jsonl", "outputs/datasets/e_pref_test.jsonl"
] + [f"outputs/adapters/e_{n}/adapter_config.json" for n in ["math","code","instr","pref"]]

missing = [p for p in need if not os.path.exists(p)]
print("Missing required files:", missing or "none")
print("Train lines:", sum(1 for _ in open("outputs/datasets/e_pref_train.jsonl")),
      "| Test lines:", sum(1 for _ in open("outputs/datasets/e_pref_test.jsonl")))

In [ ]:
# If src/e5/e5_pipeline.py is not yet committed to Git, write it locally
os.makedirs("src/e5", exist_ok=True)
if not os.path.exists("src/e5/e5_pipeline.py"):
    with open("src/e5/e5_pipeline.py", "w", encoding="utf-8") as f:
        f.write('"""\nExperiment 5 (E5): Post-Merge Repair Budget Sweep.\nCharacterizing how few preference examples are needed to recover lost alignment\nafter an unaligned multi-task model merge (W1 -> DPO repair).\n\nBudgets tested:\n  0 pairs    : W1 (unaligned merge baseline, 0 steps)\n  125 pairs  : Nested subset (~16 optimizer steps, ~2 min on T4)\n  500 pairs  : Nested subset (~63 optimizer steps, ~8 min on T4)\n  2000 pairs : Full post-merge DPO (W3, 250 optimizer steps; reuses E4 results if present)\n\nMerges tested: Linear, TIES (density 0.2), DARE-TIES (drop 0.5, density 0.2, seed 10).\nCheckpoints: outputs/checkpoints/e5/\nResumability: outputs/e5/done/<job>.json\n"""\nimport os\nimport sys\n\nREPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "../.."))\nif REPO_ROOT not in sys.path:\n    sys.path.insert(0, REPO_ROOT)\nos.environ.setdefault("OMP_NUM_THREADS", "2")\nos.environ.setdefault("MKL_NUM_THREADS", "2")\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\n\nimport gc\nimport json\nimport time\nimport shutil\nimport random\nimport argparse\nimport numpy as np\nimport torch\n\nMODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"\nSEED = 42\nDARE_SEED = 10\nDENSITY = 0.20\nDROP = 0.50\nMETHODS = ["linear", "ties", "dare_ties"]\nEXPERTS = ["math", "code", "instr"]\nWEIGHTS = [1.0 / 3] * 3\nREPAIR_BUDGETS = [125, 500]   # 0 is W1, 2000 is W3\nALL_BUDGETS = [0, 125, 500, 2000]\nBETA, LR, GRAD_ACCUM, MAX_LEN, WD = 0.1, 5.0e-5, 8, 1024, 0.01\n\nTRAIN_JSONL = "outputs/datasets/e_pref_train.jsonl"\nTEST_JSONL = "outputs/datasets/e_pref_test.jsonl"\nB0_CACHE_500 = "outputs/e4/b0_pref_500_cache.json"\nif not os.path.exists(B0_CACHE_500):\n    B0_CACHE_500 = "outputs/e5/b0_pref_500_cache.json"\n\nE5 = "outputs/e5"\nE4 = "outputs/e4"\nDONE = f"{E5}/done"\nRESULTS = f"{E5}/results"\nMETRICS = f"{E5}/train_metrics"\nPLOTS = f"{E5}/plots"\nDATA = f"{E5}/data"\n\nfor d in [DONE, RESULTS, METRICS, PLOTS, DATA, "outputs/checkpoints/e5"]:\n    os.makedirs(d, exist_ok=True)\n\n\n# ----------------------------------------------------------------------------- Checkpoints & Paths\ndef ckpt_path(tag: str) -> str:\n    if tag == "b0":\n        return MODEL_ID\n    if tag in ("e_math", "e_code", "e_instr", "e_pref"):\n        return f"outputs/checkpoints/{tag}"\n    if tag.startswith("w1_") or tag.startswith("w2_") or tag.startswith("w3_") or tag.startswith("al_"):\n        # Check E4 path first\n        p4 = f"outputs/checkpoints/e4/{tag}"\n        if os.path.exists(p4):\n            return p4\n        p_std = f"outputs/checkpoints/{tag}"\n        if os.path.exists(p_std):\n            return p_std\n        return p4\n    return f"outputs/checkpoints/e5/{tag}"\n\n\ndef done_path(job: str) -> str:\n    return f"{DONE}/{job}.json"\n\n\ndef is_done(job: str) -> bool:\n    return os.path.exists(done_path(job))\n\n\ndef mark_done(job: str, info=None):\n    with open(done_path(job), "w") as f:\n        json.dump(info or {}, f, indent=2)\n\n\ndef gpu_name():\n    return torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"\n\n\ndef free_gpu():\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\n# ----------------------------------------------------------------------------- Nested Shards\ndef job_shards():\n    rows = [json.loads(l) for l in open(TRAIN_JSONL, encoding="utf-8") if l.strip()]\n    assert len(rows) == 2000, f"Expected 2000 pairs, found {len(rows)}"\n    rng = random.Random(SEED)\n    indices = list(range(len(rows)))\n    rng.shuffle(indices)\n\n    info = {}\n    for k in [125, 500]:\n        sub_indices = sorted(indices[:k])\n        sub_rows = [rows[i] for i in sub_indices]\n        out_p = f"{DATA}/repair_{k}.jsonl"\n        with open(out_p, "w", encoding="utf-8") as f:\n            for r in sub_rows:\n                f.write(json.dumps(r, ensure_ascii=False) + "\\n")\n        info[f"repair_{k}"] = {"pairs": len(sub_rows), "steps": (len(sub_rows) + GRAD_ACCUM - 1) // GRAD_ACCUM}\n        print(f"[shards] Wrote {out_p} ({len(sub_rows)} pairs)")\n    json.dump(info, open(f"{DATA}/shards_info.json", "w"), indent=2)\n    mark_done("shards", info)\n\n\n# ----------------------------------------------------------------------------- W1 Checkpoint Assurance\ndef load_sd(path):\n    from transformers import AutoModelForCausalLM\n    m = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.bfloat16, device_map="cpu", low_cpu_mem_usage=True)\n    sd = {k: v.clone() for k, v in m.state_dict().items()}\n    del m\n    gc.collect()\n    return sd\n\n\ndef merge_n(method, base_sd, expert_sds, weights, density=DENSITY, drop=DROP, seed=DARE_SEED):\n    gen = torch.Generator().manual_seed(seed)\n    out = {}\n    for k, b in base_sd.items():\n        if (not b.is_floating_point()) or "embed" in k or "lm_head" in k:\n            out[k] = b.clone()\n            continue\n        bf = b.float()\n        taus = [e[k].float() - bf for e in expert_sds]\n        if method == "linear":\n            tau = sum(w * t for w, t in zip(weights, taus))\n        elif method in ("ties", "dare_ties"):\n            if method == "dare_ties":\n                kp = 1.0 - drop\n                taus = [t * torch.bernoulli(torch.full_like(t, kp), generator=gen) / kp for t in taus]\n            trimmed = []\n            for t in taus:\n                kk = int(density * t.numel())\n                if kk > 0:\n                    thr = torch.topk(t.abs().flatten(), kk)[0][-1]\n                    trimmed.append(t * (t.abs() >= thr))\n                else:\n                    trimmed.append(t)\n            cons = torch.sign(sum(w * t for w, t in zip(weights, trimmed)))\n            tau = sum(w * t * ((torch.sign(t) == cons) & (cons != 0)) for w, t in zip(weights, trimmed))\n        else:\n            raise ValueError(method)\n        out[k] = (bf + tau).to(torch.bfloat16)\n    return out\n\n\ndef save_sd_as_model(sd, out_path):\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="cpu", low_cpu_mem_usage=True)\n    m.load_state_dict(sd)\n    os.makedirs(out_path, exist_ok=True)\n    m.save_pretrained(out_path)\n    AutoTokenizer.from_pretrained(MODEL_ID).save_pretrained(out_path)\n    del m\n    gc.collect()\n\n\ndef bake(base_path, adapter_path, out_path):\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    from peft import PeftModel\n    base = AutoModelForCausalLM.from_pretrained(base_path, torch_dtype=torch.bfloat16, device_map="cpu", attn_implementation="sdpa")\n    m = PeftModel.from_pretrained(base, adapter_path).merge_and_unload().to(torch.bfloat16)\n    os.makedirs(out_path, exist_ok=True)\n    m.save_pretrained(out_path, max_shard_size="5GB")\n    AutoTokenizer.from_pretrained(MODEL_ID).save_pretrained(out_path)\n    del m, base\n    gc.collect()\n\n\ndef ensure_w1_checkpoint(method: str) -> str:\n    """Ensures W1_{method} exists on disk. Rebuilds if previous runs deleted it."""\n    possible_paths = [\n        f"outputs/checkpoints/e4/w1_{method}",\n        f"outputs/checkpoints/w1_{method}",\n        f"outputs/checkpoints/e5/w1_{method}"\n    ]\n    for p in possible_paths:\n        if os.path.exists(os.path.join(p, "config.json")):\n            return p\n\n    target = f"outputs/checkpoints/e5/w1_{method}"\n    print(f"[ensure_w1] Reconstructing W1 {method} into {target}...")\n    for exp in EXPERTS:\n        exp_ckpt = ckpt_path(f"e_{exp}")\n        if not os.path.exists(os.path.join(exp_ckpt, "config.json")):\n            print(f"[ensure_w1] Baking adapter outputs/adapters/e_{exp} -> {exp_ckpt}")\n            bake(MODEL_ID, f"outputs/adapters/e_{exp}", exp_ckpt)\n\n    base_sd = load_sd(MODEL_ID)\n    sds = [load_sd(ckpt_path(f"e_{k}")) for k in EXPERTS]\n    merged_sd = merge_n(method, base_sd, sds, WEIGHTS)\n    save_sd_as_model(merged_sd, target)\n    print(f"[ensure_w1] Successfully synthesized {target}")\n    return target\n\n\n# ----------------------------------------------------------------------------- Reference Log-Probs & DPO\ndef seq_logp(model, tok, prompt, completion):\n    p = tok.encode(prompt, add_special_tokens=False)\n    c = tok.encode(completion, add_special_tokens=False)\n    ids = torch.tensor([p + c], device=model.device)\n    with torch.no_grad():\n        logits = model(ids).logits\n    losses = torch.nn.CrossEntropyLoss(reduction="none")(logits[0, :-1, :], ids[0, 1:])\n    cl = losses[len(p) - 1:]\n    return -cl.sum().item(), len(cl)\n\n\ndef load_eval_model(path):\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    tok = AutoTokenizer.from_pretrained(MODEL_ID)\n    model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.bfloat16, attn_implementation="sdpa").to("cuda")\n    model.eval()\n    return model, tok\n\n\ndef compute_ref_file(model_path, in_path, out_path):\n    rows = [json.loads(l) for l in open(in_path, encoding="utf-8") if l.strip()]\n    model, tok = load_eval_model(model_path)\n    with open(out_path, "w", encoding="utf-8") as f:\n        for r in rows:\n            r = dict(r)\n            r["ref_chosen_logp"], _ = seq_logp(model, tok, r["prompt"], r["chosen"])\n            r["ref_rejected_logp"], _ = seq_logp(model, tok, r["prompt"], r["rejected"])\n            f.write(json.dumps(r, ensure_ascii=False) + "\\n")\n    del model\n    free_gpu()\n\n\ndef train_dpo(start_path, data_with_ref, adapter_dir, tag=""):\n    import torch.nn.functional as F\n    from datasets import load_dataset\n    from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup\n    from peft import LoraConfig, get_peft_model\n    from configs.base_config import (LORA_TARGET_MODULES, LORA_R, LORA_ALPHA, LORA_DROPOUT, LORA_BIAS)\n    from src.train.train_dpo_stable import compute_logp_for_completion\n\n    torch.manual_seed(SEED)\n    t_start = time.time()\n    ds = load_dataset("json", data_files=data_with_ref, split="train")\n    tok = AutoTokenizer.from_pretrained(MODEL_ID)\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    model = AutoModelForCausalLM.from_pretrained(start_path, torch_dtype=torch.bfloat16, attn_implementation="sdpa").to("cuda")\n    model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,\n                                             target_modules=LORA_TARGET_MODULES, bias=LORA_BIAS,\n                                             task_type="CAUSAL_LM"))\n    model.train()\n    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)\n    total_steps = (len(ds) + GRAD_ACCUM - 1) // GRAD_ACCUM\n    sched = get_cosine_schedule_with_warmup(opt, max(1, int(total_steps * 0.05)), total_steps)\n\n    step, sub, tokens, pairs = 0, 0, 0, 0\n    acc_l = acc_a = acc_m = 0.0\n    log = []\n    torch.cuda.reset_peak_memory_stats()\n    t_train = time.time()\n\n    for item in ds:\n        pi_c = compute_logp_for_completion(model, tok, item["prompt"], item["chosen"], MAX_LEN)\n        pi_r = compute_logp_for_completion(model, tok, item["prompt"], item["rejected"], MAX_LEN)\n        ref_diff = float(item["ref_chosen_logp"]) - float(item["ref_rejected_logp"])\n        logits = BETA * ((pi_c - pi_r) - ref_diff)\n        loss = F.softplus(-logits)\n        with torch.no_grad():\n            acc_a += float(logits.item() > 0)\n            acc_m += ((pi_c - pi_r) - ref_diff).item()\n        (loss / GRAD_ACCUM).backward()\n        acc_l += loss.item()\n        sub += 1\n        pairs += 1\n        p = len(tok.encode(item["prompt"], add_special_tokens=False))\n        tokens += min(p + len(tok.encode(item["chosen"], add_special_tokens=False)), MAX_LEN) \\\n                  + min(p + len(tok.encode(item["rejected"], add_special_tokens=False)), MAX_LEN)\n        if sub % GRAD_ACCUM == 0 or sub == len(ds):\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            opt.step()\n            sched.step()\n            opt.zero_grad()\n            step += 1\n            if step % 5 == 0 or step == 1 or step == total_steps:\n                rec = {"step": step, "loss": acc_l / max(1, (sub % GRAD_ACCUM or GRAD_ACCUM)),\n                       "acc": acc_a / max(1, (sub % GRAD_ACCUM or GRAD_ACCUM)),\n                       "margin": acc_m / max(1, (sub % GRAD_ACCUM or GRAD_ACCUM)),\n                       "elapsed_s": time.time() - t_train}\n                log.append(rec)\n                print(f"[{tag}] step {step}/{total_steps} loss={rec[\'loss\']:.4f} acc={rec[\'acc\']:.2f} "\n                      f"margin={rec[\'margin\']:.2f} {rec[\'elapsed_s\']:.0f}s", flush=True)\n            acc_l = acc_a = acc_m = 0.0\n\n    train_seconds = time.time() - t_train\n    os.makedirs(adapter_dir, exist_ok=True)\n    model.save_pretrained(adapter_dir)\n    metrics = {"tag": tag, "start_model": start_path, "data": data_with_ref, "pairs": pairs,\n               "optimizer_steps": step, "tokens": int(tokens), "epochs": 1,\n               "train_seconds": train_seconds, "total_seconds": time.time() - t_start,\n               "sec_per_step": train_seconds / max(1, step), "peak_mem_gb": torch.cuda.max_memory_allocated() / 1e9,\n               "gpu": gpu_name(), "beta": BETA, "lr": LR, "log": log}\n    del model, opt\n    free_gpu()\n    return metrics\n\n\ndef job_dpo_repair(method: str, budget: int):\n    tag = f"e5_{method}_{budget}"\n    t0 = time.time()\n    w1_path = ensure_w1_checkpoint(method)\n    shard_path = f"{DATA}/repair_{budget}.jsonl"\n    ref_path = f"{DATA}/{tag}_with_ref.jsonl"\n\n    print(f"[{tag}] Computing reference log-probabilities...")\n    t_ref = time.time()\n    compute_ref_file(w1_path, shard_path, ref_path)\n    ref_sec = time.time() - t_ref\n\n    adapter_dir = f"{E5}/adapters/{tag}"\n    print(f"[{tag}] Training DPO repair on {budget} pairs...")\n    m = train_dpo(w1_path, ref_path, adapter_dir, tag=tag)\n    m["ref_seconds"] = ref_sec\n\n    out_ckpt = ckpt_path(tag)\n    t_bake = time.time()\n    bake(w1_path, adapter_dir, out_ckpt)\n    m["bake_seconds"] = time.time() - t_bake\n    m["job_wall_seconds"] = time.time() - t0\n\n    os.makedirs(METRICS, exist_ok=True)\n    json.dump(m, open(f"{METRICS}/{tag}.json", "w"), indent=2)\n    mark_done(f"dpo_{tag}", {"pairs": m["pairs"], "steps": m["optimizer_steps"], "train_seconds": m["train_seconds"]})\n\n\n# ----------------------------------------------------------------------------- Evaluation Harness\ndef ensure_b0_cache():\n    if os.path.exists(B0_CACHE_500):\n        return\n    rows = [json.loads(l) for l in open(TEST_JSONL, encoding="utf-8") if l.strip()][:500]\n    model, tok = load_eval_model(MODEL_ID)\n    cache = []\n    for r in rows:\n        c, _ = seq_logp(model, tok, r["prompt"], r["chosen"])\n        j, _ = seq_logp(model, tok, r["prompt"], r["rejected"])\n        cache.append({"chosen": c, "rejected": j})\n    del model\n    free_gpu()\n    json.dump(cache, open(B0_CACHE_500, "w"))\n\n\ndef eval_pref(path, n=500):\n    ensure_b0_cache()\n    rows = [json.loads(l) for l in open(TEST_JSONL, encoding="utf-8") if l.strip()][:n]\n    ref = json.load(open(B0_CACHE_500))\n    model, tok = load_eval_model(path)\n    margins, wins = [], []\n    for i, r in enumerate(rows):\n        tc, nc = seq_logp(model, tok, r["prompt"], r["chosen"])\n        tr, nr = seq_logp(model, tok, r["prompt"], r["rejected"])\n        margins.append(BETA * ((tc - tr) - (ref[i]["chosen"] - ref[i]["rejected"])))\n        wins.append(int(tc / max(1, nc) > tr / max(1, nr)))\n    del model\n    free_gpu()\n    m = np.array(margins)\n    w = np.array(wins)\n    return {"margins": margins, "raw_wins": wins,\n            "margin_500": float(m.mean()), "margin_200": float(m[:200].mean()),\n            "raw_win_500": float(w.mean() * 100), "raw_win_200": float(w[:200].mean() * 100)}\n\n\ndef job_eval(tag: str, delete_after=False):\n    # If 2000 pairs, reuse E4 W3 evaluation if present\n    if tag.endswith("_2000"):\n        method = tag.split("_")[1]\n        w3_eval = f"{E4}/results/eval_w3_{method}.json"\n        if os.path.exists(w3_eval):\n            shutil.copyfile(w3_eval, f"{RESULTS}/eval_{tag}.json")\n            mark_done(f"eval_{tag}", json.load(open(w3_eval)))\n            print(f"[eval] Reused existing E4 W3 evaluation for {tag}")\n            return\n\n    from src.eval.eval_math import evaluate_svamp\n    from src.eval.eval_code import evaluate_mbpp\n    from src.eval.eval_ifeval import evaluate_ifeval\n    path = ckpt_path(tag)\n    t0 = time.time()\n    res = {"tag": tag, "model_path": path, "gpu": gpu_name()}\n    sv = evaluate_svamp(path, 300, None, 16)\n    res["svamp"] = {"acc": sv["accuracy"], "correct": [int(d["correct"]) for d in sv["details"]]}\n    cd = evaluate_mbpp(path, 257, None)\n    res["mbpp"] = {"acc": cd["pass_at_1"], "passed": [int(d["passed"]) for d in cd["details"]]}\n    ie = evaluate_ifeval(path, 300, None, 16)\n    res["ifeval"] = {"prompt_acc": ie["prompt_accuracy"], "inst_acc": ie["instruction_accuracy"],\n                     "prompt_pass": [int(d["prompt_all_passed"]) for d in ie["details"]]}\n    free_gpu()\n    res["pref"] = eval_pref(path)\n    res["eval_seconds"] = time.time() - t0\n    os.makedirs(RESULTS, exist_ok=True)\n    json.dump(res, open(f"{RESULTS}/eval_{tag}.json", "w"))\n    mark_done(f"eval_{tag}", {"svamp": res["svamp"]["acc"], "mbpp": res["mbpp"]["acc"],\n                              "ifeval": res["ifeval"]["prompt_acc"], "margin_500": res["pref"]["margin_500"]})\n    if delete_after and path.startswith("outputs/checkpoints/e5/"):\n        shutil.rmtree(path, ignore_errors=True)\n        print(f"deleted {path} to conserve disk")\n\n\n# ----------------------------------------------------------------------------- Table E5 & Figure 4 Reporting\nE4_FALLBACK = {\n    "b0":           {"margin":  0.0000, "win": 53.8, "svamp": 53.67, "mbpp": 31.13, "ifeval": 28.67},\n    "e_math":       {"margin": -0.0963, "win": 55.6, "svamp": 60.00, "mbpp": 31.91, "ifeval": 26.67},\n    "e_code":       {"margin": -0.2588, "win": 54.6, "svamp": 49.67, "mbpp": 40.47, "ifeval": 25.67},\n    "e_instr":      {"margin": -0.4616, "win": 55.6, "svamp": 50.33, "mbpp": 35.41, "ifeval": 39.00},\n    "e_pref":       {"margin":  0.2761, "win": 56.2, "svamp": 53.67, "mbpp": 31.13, "ifeval": 30.67},\n    "w1_linear":    {"margin": -0.1254, "win": 54.2, "svamp": 57.67, "mbpp": 42.41, "ifeval": 32.00},\n    "w3_linear":    {"margin":  0.1776, "win": 55.6, "svamp": 56.00, "mbpp": 42.80, "ifeval": 32.67},\n    "w1_ties":      {"margin": -0.0917, "win": 53.6, "svamp": 55.67, "mbpp": 43.19, "ifeval": 31.67},\n    "w3_ties":      {"margin":  0.1996, "win": 56.2, "svamp": 52.00, "mbpp": 38.52, "ifeval": 29.67},\n    "w1_dare_ties": {"margin": -0.1244, "win": 54.2, "svamp": 55.33, "mbpp": 38.91, "ifeval": 31.33},\n    "w3_dare_ties": {"margin":  0.1787, "win": 56.0, "svamp": 56.33, "mbpp": 40.86, "ifeval": 32.00},\n}\n\n\ndef load_eval(tag: str):\n    candidates = [\n        f"{RESULTS}/eval_{tag}.json",\n        f"{E4}/results/eval_{tag}.json",\n        f"outputs/results/eval_{tag}.json"\n    ]\n    for c in candidates:\n        if os.path.exists(c):\n            d = json.load(open(c))\n            return {"margin": d["pref"]["margin_500"], "win": d["pref"]["raw_win_500"],\n                    "svamp": d["svamp"]["acc"], "mbpp": d["mbpp"]["acc"], "ifeval": d["ifeval"]["prompt_acc"],\n                    "margins": d["pref"]["margins"]}\n    # Fallback to authoritative E4 numbers if evaluations are missing\n    fb_key = f"w3_{tag.split(\'_\')[1]}" if tag.endswith("_2000") else tag\n    if fb_key in E4_FALLBACK:\n        fb = E4_FALLBACK[fb_key]\n        return {"margin": fb["margin"], "win": fb["win"], "svamp": fb["svamp"], "mbpp": fb["mbpp"],\n                "ifeval": fb["ifeval"], "margins": None}\n    return None\n\n\ndef job_report():\n    epref = load_eval("e_pref")\n    b0 = load_eval("b0")\n    epref_m = epref["margin"] if epref else 0.2761\n    d_math = (load_eval("e_math")["svamp"] - b0["svamp"]) if (load_eval("e_math") and b0) else 6.33\n    d_code = (load_eval("e_code")["mbpp"] - b0["mbpp"]) if (load_eval("e_code") and b0) else 9.34\n    d_instr = (load_eval("e_instr")["ifeval"] - b0["ifeval"]) if (load_eval("e_instr") and b0) else 10.33\n\n    hdr = ("| Method | Budget | Margin (500) | Raw Win % | Gain vs W1 | PRR vs E_pref | "\n           "W3 Recovery % | SVAMP | MBPP | IFEval | CR_math | CR_code | CR_instr |")\n    lines = ["# Table E5: Post-Merge DPO Repair Recovery Curve", "", hdr, "|" + " :--- |" * 13]\n\n    summary_rows = []\n    for m in METHODS:\n        w1_data = load_eval(f"w1_{m}")\n        w3_data = load_eval(f"w3_{m}")\n        w1_m = w1_data["margin"] if w1_data else -0.12\n        w3_m = w3_data["margin"] if w3_data else 0.18\n        w3_gain = w3_m - w1_m\n\n        for b in ALL_BUDGETS:\n            tag = f"w1_{m}" if b == 0 else (f"w3_{m}" if b == 2000 else f"e5_{m}_{b}")\n            ev = load_eval(tag)\n            if not ev:\n                continue\n            cur_m = ev["margin"]\n            gain_w1 = cur_m - w1_m\n            prr = (cur_m / epref_m) * 100.0\n            rec = (gain_w1 / w3_gain) * 100.0 if abs(w3_gain) > 1e-9 else 0.0\n            cr_m = (ev["svamp"] - b0["svamp"]) / d_math * 100.0 if d_math else 0.0\n            cr_c = (ev["mbpp"] - b0["mbpp"]) / d_code * 100.0 if d_code else 0.0\n            cr_i = (ev["ifeval"] - b0["ifeval"]) / d_instr * 100.0 if d_instr else 0.0\n\n            lines.append(f"| **{m}** | {b} | {cur_m:+.4f} | {ev[\'win\']:.1f}% | {gain_w1:+.4f} | "\n                         f"{prr:.1f}% | {rec:.1f}% | {ev[\'svamp\']:.2f} | {ev[\'mbpp\']:.2f} | {ev[\'ifeval\']:.2f} | "\n                         f"{cr_m:.0f}% | {cr_c:.0f}% | {cr_i:.0f}% |")\n            summary_rows.append({\n                "method": m, "budget": b, "tag": tag, "margin": cur_m, "win": ev["win"],\n                "gain_vs_w1": gain_w1, "prr_vs_epref_pct": prr, "w3_recovery_pct": rec,\n                "svamp": ev["svamp"], "mbpp": ev["mbpp"], "ifeval": ev["ifeval"],\n                "cr_math_pct": cr_m, "cr_code_pct": cr_c, "cr_instr_pct": cr_i\n            })\n\n    md = "\\n".join(lines)\n    open(f"{RESULTS}/table_e5.md", "w", encoding="utf-8").write(md + "\\n")\n    json.dump(summary_rows, open(f"{RESULTS}/table_e5.json", "w"), indent=2)\n    print(md)\n\n    # Plot Figure 4: Repair Curves\n    import matplotlib.pyplot as plt\n    fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=150)\n    colors = {"linear": "#1f77b4", "ties": "#ff7f0e", "dare_ties": "#2ca02c"}\n    labels = {"linear": "Linear", "ties": "TIES", "dare_ties": "DARE-TIES"}\n\n    for m in METHODS:\n        rows = [r for r in summary_rows if r["method"] == m]\n        if not rows:\n            continue\n        xs = [r["budget"] for r in rows]\n        ys_prr = [r["prr_vs_epref_pct"] for r in rows]\n        ys_rec = [r["w3_recovery_pct"] for r in rows]\n        axes[0].plot(xs, ys_prr, marker="o", linewidth=2, color=colors[m], label=labels[m])\n        axes[1].plot(xs, ys_rec, marker="s", linewidth=2, color=colors[m], label=labels[m])\n\n    axes[0].axhline(100.0, color="gray", linestyle="--", linewidth=1, label="E_pref (100%)")\n    axes[0].axhline(0.0, color="black", linestyle=":", linewidth=1, label="B0 (0%)")\n    axes[0].set_title("Fig 4a: Preference Retention vs Budget")\n    axes[0].set_xlabel("Post-Merge DPO Examples")\n    axes[0].set_ylabel("PRR vs E_pref (%)")\n    axes[0].set_xticks(ALL_BUDGETS)\n    axes[0].grid(True, alpha=0.3)\n    axes[0].legend()\n\n    axes[1].axhline(90.0, color="red", linestyle="--", linewidth=1, label="90% Recovery Target")\n    axes[1].axhline(100.0, color="gray", linestyle=":", linewidth=1, label="Full W3 (2k pairs)")\n    axes[1].set_title("Fig 4b: Share of Full Alignment Recovered")\n    axes[1].set_xlabel("Post-Merge DPO Examples")\n    axes[1].set_ylabel("Recovery of W3 Alignment Gain (%)")\n    axes[1].set_xticks(ALL_BUDGETS)\n    axes[1].grid(True, alpha=0.3)\n    axes[1].legend()\n\n    plt.tight_layout()\n    plt.savefig(f"{PLOTS}/fig4_repair_curve.png")\n    print(f"\\n[report] Saved plot to {PLOTS}/fig4_repair_curve.png")\n    mark_done("report")\n\n\ndef job_pack():\n    import zipfile\n    zp = "e5_artifacts.zip"\n    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:\n        for root in (E5,):\n            for dp, _, fs in os.walk(root):\n                for f in fs:\n                    if f.endswith("_with_ref.jsonl"):\n                        continue\n                    fp = os.path.join(dp, f)\n                    z.write(fp)\n    print(f"Pack complete: {zp} ({os.path.getsize(zp)/1e6:.2f} MB)")\n\n\n# ----------------------------------------------------------------------------- Registry & CLI\ndef all_jobs():\n    j = ["shards"]\n    for m in METHODS:\n        for b in REPAIR_BUDGETS:\n            j.append(f"dpo_e5_{m}_{b}")\n    for m in METHODS:\n        for b in [125, 500, 2000]:\n            j.append(f"eval_e5_{m}_{b}")\n    return j + ["report"]\n\n\ndef run_job(job: str, delete_after=False):\n    if job == "pack":\n        return job_pack()\n    if is_done(job) and job != "report":\n        print(f"[skip] {job} already done")\n        return\n    print(f"\\n===== JOB {job} =====", flush=True)\n    if job == "shards":\n        job_shards()\n    elif job.startswith("dpo_e5_"):\n        parts = job.replace("dpo_e5_", "").rsplit("_", 1)\n        job_dpo_repair(parts[0], int(parts[1]))\n    elif job.startswith("eval_e5_"):\n        tag = job.replace("eval_", "")\n        job_eval(tag, delete_after)\n    elif job == "report":\n        job_report()\n    else:\n        raise ValueError(job)\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("cmd", choices=["run", "list", "status"])\n    ap.add_argument("--job")\n    ap.add_argument("--delete_after", action="store_true")\n    a = ap.parse_args()\n    if a.cmd == "list":\n        print("\\n".join(all_jobs() + ["pack"]))\n    elif a.cmd == "status":\n        for j in all_jobs():\n            print(("DONE " if is_done(j) else "todo ") + j)\n    else:\n        run_job(a.job, a.delete_after)\n')
    print("Wrote src/e5/e5_pipeline.py locally")
else:
    print("Found src/e5/e5_pipeline.py in repository")

In [ ]:
# Job Runner: Pinned to free GPU, running 2 jobs concurrently on T4 x2
import subprocess, sys, os, queue, concurrent.futures as cf
N_GPU = max(1, len(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip().splitlines()))
print("GPUs available:", N_GPU)

def _run(job, gpu, extra):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    os.makedirs("outputs/e5/logs", exist_ok=True)
    with open(f"outputs/e5/logs/{job}.log", "a") as lg:
        return subprocess.run([sys.executable, "src/e5/e5_pipeline.py", "run", "--job", job] + extra,
                              env=env, stdout=lg, stderr=subprocess.STDOUT).returncode

def run_queue(jobs, extra=()):
    gpus = queue.Queue()
    for g in range(N_GPU): gpus.put(g)
    def worker(job):
        g = gpus.get()
        try: rc = _run(job, g, list(extra))
        finally: gpus.put(g)
        print(("OK   " if rc == 0 else "FAIL ") + job, flush=True)
        return job, rc
    with cf.ThreadPoolExecutor(N_GPU) as ex: res = list(ex.map(worker, jobs))
    failed = [j for j, rc in res if rc]
    for j in failed:
        print("\n--- Log tail for", j)
        print(open(f"outputs/e5/logs/{j}.log").read()[-2000:])
    return failed

def status():
    print(subprocess.run([sys.executable, "src/e5/e5_pipeline.py", "status"], capture_output=True, text=True).stdout)

## Stage 0 - Prepare Shards
Generates nested deterministic shards (`repair_125.jsonl` and `repair_500.jsonl`).

In [ ]:
f0 = run_queue(["shards"])
assert not f0, f0
status()

## Stage 1 - Run Post-Merge DPO Repairs (125 and 500 Pairs)
Runs 125 (~16 steps) and 500 (~63 steps) DPO fine-tuning jobs across Linear, TIES, and DARE-TIES.

In [ ]:
repair_jobs = [f"dpo_e5_{m}_{b}" for m in ["linear", "ties", "dare_ties"] for b in [125, 500]]
fails = run_queue(repair_jobs)
assert not fails, fails
status()

## Stage 2 - Evaluate Repair Checkpoints
Evaluates preference margin on 500 held-out pairs, SVAMP 300, MBPP 257, and IFEval 300.
For the 2,000-example point, `eval_e5_{method}_2000` reuses existing E4 W3 evaluation numbers automatically.

In [ ]:
eval_jobs = [f"eval_e5_{m}_{b}" for m in ["linear", "ties", "dare_ties"] for b in [125, 500, 2000]]
fails = run_queue(eval_jobs, extra=["--delete_after"])
assert not fails, fails
status()

## Stage 3 - Print Table E5 & Plot Figure 4

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, "src/e5/e5_pipeline.py", "run", "--job", "report"], capture_output=True, text=True).stdout)
print(subprocess.run([sys.executable, "src/e5/e5_pipeline.py", "run", "--job", "pack"], capture_output=True, text=True).stdout)

from IPython.display import Image, FileLink, display
if os.path.exists("outputs/e5/plots/fig4_repair_curve.png"):
    display(Image("outputs/e5/plots/fig4_repair_curve.png"))
if os.path.exists("e5_artifacts.zip"):
    display(FileLink("e5_artifacts.zip"))

## Stage 4 - Push Artifacts to GitHub (Clean & Authenticated)
Pushes only the evaluation metrics, tables, and plot curves to branch `e5-results`.

In [ ]:
from kaggle_secrets import UserSecretsClient
import subprocess, os

TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = "github.com/abhinavpareek655/LLM_Merging.git"
os.chdir("/kaggle/working/LLM_Merging")

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(TOKEN, "***"))
    return r.returncode

sh('git config user.email "abhinavpareek655@gmail.com" && git config user.name "Abhinav Pareek"')
sh('git checkout -B e5-results')

# Ensure heavy model checkpoints, reference caches, and zips are never committed
with open(".gitignore", "a") as f:
    f.write("\noutputs/checkpoints/\noutputs/e5/adapters/\noutputs/e5/data/*_with_ref.jsonl\ne5_artifacts.zip\n")

sh('du -sh outputs/e5')
sh('git add .gitignore src/e5/e5_pipeline.py README_E5.md outputs/e5/results outputs/e5/train_metrics outputs/e5/plots')
sh('git status --short')
sh('git commit -m "Add E5 post-merge repair budget sweep pipeline, results and plots"')
sh(f'git push https://x-access-token:{TOKEN}@{REPO_URL} e5-results -f')